In [1]:
#_________________TASK 1_________________

In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta
from pprint import pprint
import random
from collections import deque, defaultdict, Counter

np.random.seed(42)
n=200

products = ["Laptop", "Phone", "Keyboard", "Mouse", "Monitor", "Tablet"]
countries = ["USA", "Germany", "France", "UK", "Canada", "Japan"]

raw_orders = []

start_date = datetime(2025, 1, 1)

for i in range(200):
    order = {
        "order_id": 1000 + i,
        "customer_id": np.random.randint(1, 50),
        "product_name": np.random.choice(products),
        "quantity": np.random.randint(1, 5),
        "unit_price": round(random.uniform(10, 150), 2),
        "order_date": (start_date + timedelta(days=random.randint(0, 180))).strftime("%Y-%m-%d"),
        "shipping_country": np.random.choice(countries)
    }
    raw_orders.append(order)

raw_orders[5]["product_name"] = None
raw_orders[105]["product_name"] = ""
raw_orders[199]["product_name"] = None

raw_orders[10]["unit_price"] = -10
raw_orders[50]["quantity"] = -20
raw_orders[150]["quantity"] = -15

raw_orders[12]["order_date"] = "not-a-date"
raw_orders[24]["order_date"] = "2025-13-45"
raw_orders[36]["order_date"] = "not-a-date"

raw_orders[48]["order_id"] = raw_orders[1]["order_id"]
raw_orders[56]["order_id"] = raw_orders[2]["order_id"]
raw_orders[69]["order_id"] = raw_orders[3]["order_id"]

raw_orders[72]["unit_price"] = str(raw_orders[10]["unit_price"])
raw_orders[84]["unit_price"] = str(raw_orders[84]["unit_price"])
raw_orders[92]["quantity"] = str(raw_orders[92]["quantity"])


def extract(raw_records):
    valid_records = []
    rejected_records = []
    rejection_stats = defaultdict(int)

    required_fields = [
        "order_id", 
        "customer_id", 
        "product_name", 
        "quantity", 
        "unit_price", 
        "order_date", 
        "shipping_country"
    ]

    seen_order_ids = set()
    
    for record in raw_records:
        reason = None

        for field in required_fields:
            if field not in record or record[field] in (None, ""):
                reason = f"missing_{field}"
                break

        if not reason:
            if record["order_id"] in seen_order_ids:
                reason = "duplicate_order_id"
            else:
                seen_order_ids.add(record["order_id"])

        if not reason:
            if not isinstance(record["quantity"], (int, float)):
                reason = "invalid_quantity_type"
            elif not isinstance(record["unit_price"], (int, float)):
                reason = "invalid_price_type"

        if not reason:
            if record["quantity"] <= 0 or record["unit_price"] <= 0:
                reason = "negative_or_zero_values"

        if not reason:
            try:
                datetime.strptime(record["order_date"], "%Y-%m-%d")
            except (ValueError, TypeError):
                reason = "invalid_date"

        if reason:
            rejected = record.copy()
            rejected["rejection_reason"] = reason
            rejected_records.append(rejected)
            rejection_stats[reason] += 1
        else:
            valid_records.append(record)

    return valid_records, rejected_records, rejection_stats

valid, rejected, stats = extract(raw_orders)

print(f"Valid Records: {len(valid)}")
print(f"Rejected Records: {len(rejected)}")
print("")
print("Rejection Breakdown:")
for reason, count in stats.items():
    print(f" - {reason}: {count}")

print("")

def transform(valid_records):
    df = pd.DataFrame(valid_records)

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    df["total_amount"] = df["quantity"] * df["unit_price"]

    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    
    df["order_month"] = df["order_date"].dt.month
    df["order_day_of_week"] = df["order_date"].dt.day_name()

    df["shipping_country"] = df["shipping_country"].str.title()

    df = df.drop_duplicates(subset=["order_id"], keep="first")

    df["amount_category"] = "large"
    df.loc[df["total_amount"] < 25, "amount_category"] = "small"
    df.loc[(df["total_amount"] >= 25) & (df["total_amount"] <= 100), "amount_category"] = "medium"

    df = df.reset_index(drop=True)
    
    return df

clean_df = transform(valid)
print(clean_df.head())

print("")

def load(df, path):
    df.to_parquet(path, engine="pyarrow", index=False)
    print(f"Saved DataFrame to {path}")

    df_loaded = pd.read_parquet(path, engine="pyarrow")

    if len(df) == len(df_loaded):
        print(f"Row count matches: {len(df)} rows")
    else:
        print(f"Row count mismatch! Original: {len(df)}, Loaded: {len(df_loaded)}")

    print("")
    
    print("Column dtypes check:")
    print(pd.DataFrame({
        "original_dtype": df.dtypes,
        "loaded_dtype": df_loaded.dtypes
    }))

    return df_loaded

parquet_path = "clean_orders.parquet"

df_loaded = load(clean_df, parquet_path)

print("\nPipeline Summary")
print(f"Raw records: {len(raw_orders)}")
print(f"Valid records: {len(valid)}")
print(f"Rejected records: {len(rejected)}")
print(f"Transformed records: {len(clean_df)}")
print(f"Loaded records: {len(df_loaded)}")

Valid Records: 185
Rejected Records: 15

Rejection Breakdown:
 - missing_product_name: 3
 - negative_or_zero_values: 3
 - invalid_date: 3
 - duplicate_order_id: 3
 - invalid_price_type: 2
 - invalid_quantity_type: 1

   order_id  customer_id product_name  quantity  unit_price order_date  \
0      1000           39        Mouse         1      110.63 2025-01-13   
1      1001            8      Monitor         1       28.34 2025-03-11   
2      1002           19     Keyboard         3       72.58 2025-03-09   
3      1003           36     Keyboard         2       59.54 2025-06-08   
4      1004            2        Mouse         2      120.88 2025-02-10   

  shipping_country  total_amount  order_month order_day_of_week  \
0           France        110.63            1            Monday   
1          Germany         28.34            3           Tuesday   
2           Canada        217.74            3            Sunday   
3           Canada        119.08            6            Sunday   
4  

NameError: name 'df_loaded' is not defined

In [ ]:
#_________________TASK 2_________________

In [ ]:
np.random.seed(42)
n=200

products = ["Laptop", "Phone", "Keyboard", "Mouse", "Monitor", "Tablet"]
countries = ["USA", "Germany", "France", "UK", "Canada", "Japan"]

raw_orders_new = []
start_date = datetime(2025, 1, 1)

for i in range(n):
    order = {
        "order_id": 2000 + i,
        "customer_id": np.random.randint(1, 50),
        "product_name": np.random.choice(products),
        "quantity": np.random.randint(1, 5),
        "unit_price": round(random.uniform(10, 150), 2),
        "order_date": (start_date + timedelta(days=np.random.randint(0, 180))).strftime("%Y-%m-%d"),
        "shipping_country": np.random.choice(countries)
    }
    raw_orders_new.append(order)

raw_orders_new[5]["product_name"] = None
raw_orders_new[105]["product_name"] = ""
raw_orders_new[199]["product_name"] = None

raw_orders_new[10]["unit_price"] = -10
raw_orders_new[50]["quantity"] = -20
raw_orders_new[150]["quantity"] = -15

raw_orders_new[12]["order_date"] = "not-a-date"
raw_orders_new[24]["order_date"] = "2025-13-45"
raw_orders_new[36]["order_date"] = "not-a-date"

raw_orders_new[48]["order_id"] = raw_orders_new[1]["order_id"]
raw_orders_new[56]["order_id"] = raw_orders_new[2]["order_id"]
raw_orders_new[69]["order_id"] = raw_orders_new[3]["order_id"]

raw_orders_new[72]["unit_price"] = str(raw_orders_new[10]["unit_price"])
raw_orders_new[84]["unit_price"] = str(raw_orders_new[84]["unit_price"])
raw_orders_new[92]["quantity"] = str(raw_orders_new[92]["quantity"])

print(len(raw_orders_new))

minimal_extracted_new = [r for r in raw_orders_new if r is not None]
print(len(minimal_extracted_new))

df_minimal_new = pd.DataFrame(minimal_extracted_new)
df_minimal_new["quantity"] = pd.to_numeric(df_minimal_new["quantity"], errors="coerce")
df_minimal_new["unit_price"] = pd.to_numeric(df_minimal_new["unit_price"], errors="coerce")

data_lake_path_new = "raw_orders_data_lake_new.parquet"
df_minimal_new.to_parquet(data_lake_path_new, engine="pyarrow", index=False)
print(f"Raw data loaded to Parquet: {data_lake_path_new}")

def extract(raw_records_new):
    valid_records_new = []
    rejected_records_new = []
    rejection_stats_new = defaultdict(int)

    required_fields_new = ["order_id", "customer_id", "product_name",
                           "quantity", "unit_price", "order_date", "shipping_country"]
    seen_order_ids_new = set()

    for record in raw_records_new:
        reason_new = None
        for field in required_fields_new:
            if field not in record or record[field] in (None, ""):
                reason_new = f"missing_{field}"
                break

        if not reason_new:
            if record["order_id"] in seen_order_ids_new:
                reason_new = "duplicate_order_id"
            else:
                seen_order_ids_new.add(record["order_id"])

        if not reason_new:
            if not isinstance(record["quantity"], (int, float)):
                reason_new = "invalid_quantity_type"
            elif not isinstance(record["unit_price"], (int, float)):
                reason_new = "invalid_price_type"

        if not reason_new:
            if record["quantity"] <= 0 or record["unit_price"] <= 0:
                reason_new = "negative_or_zero_values"

        if not reason_new:
            try:
                datetime.strptime(record["order_date"], "%Y-%m-%d")
            except (ValueError, TypeError):
                reason_new = "invalid_date"

        if reason_new:
            rejected_new = record.copy()
            rejected_new["rejection_reason"] = reason_new
            rejected_records_new.append(rejected_new)
            rejection_stats_new[reason_new] += 1
        else:
            valid_records_new.append(record)

    return valid_records_new, rejected_records_new, rejection_stats_new

def transform(valid_records_new):
    df_transformed_new = pd.DataFrame(valid_records_new)
    
    df_transformed_new["quantity"] = pd.to_numeric(df_transformed_new["quantity"], errors="coerce")
    df_transformed_new["unit_price"] = pd.to_numeric(df_transformed_new["unit_price"], errors="coerce")
    
    df_transformed_new["total_amount"] = df_transformed_new["quantity"] * df_transformed_new["unit_price"]

    df_transformed_new["order_date"] = pd.to_datetime(df_transformed_new["order_date"], errors="coerce")
    
    df_transformed_new["order_month"] = df_transformed_new["order_date"].dt.month
    df_transformed_new["order_day_of_week"] = df_transformed_new["order_date"].dt.day_name()
    df_transformed_new["shipping_country"] = df_transformed_new["shipping_country"].str.title()
    
    df_transformed_new = df_transformed_new.drop_duplicates(subset=["order_id"], keep="first")
    
    df_transformed_new["amount_category"] = "large"
    df_transformed_new.loc[df_transformed_new["total_amount"] < 25, "amount_category"] = "small"
    df_transformed_new.loc[(df_transformed_new["total_amount"] >= 25) & (df_transformed_new["total_amount"] <= 100), "amount_category"] = "medium"
    
    return df_transformed_new.reset_index(drop=True)

def load(df_transformed_new, parquet_path_new):
    df_transformed_new.to_parquet(parquet_path_new, engine="pyarrow", index=False)
    print(f"Saved DataFrame to {parquet_path_new}")
    df_loaded_new = pd.read_parquet(parquet_path_new, engine="pyarrow")
    print(f"Loaded {len(df_loaded_new)} rows")
    return df_loaded_new

valid_new, rejected_new, stats_new = extract(minimal_extracted_new)
print(f"\nValid Records: {len(valid_new)}")
print(f"Rejected Records: {len(rejected_new)}")
print("Rejection breakdown:")
for reason, count in stats_new.items():
    print(f" - {reason}: {count}")

df_transformed_new = transform(valid_new)
print("\nTransformed DataFrame preview:")
print(df_transformed_new.head())

parquet_path_transformed = "clean_orders_new.parquet"
df_loaded_new = load(df_transformed_new, parquet_path_transformed)

In [ ]:
"""
In the ETL approach, only the valid records are saved to the destination. All invalid or duplicate records are removed before loading.

In the ELT approach, all raw records are saved first to the data lake. Problems are found later, during the transformation stage.

ETL is good because the data is clean and ready to use immediately. ELT is good because all raw data is kept and can be reprocessed later.

You would choose ETL when you need clean data right away. You would choose ELT when you want to keep all raw data and fix or analyze it later.
"""

In [ ]:
#_________________TASK 3_________________

In [ ]:
database = {
    "orders": [],
    "features": {}
}

def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():
    features = {}

    for order in database["orders"]:
        cid = order["customer_id"]
        amount = order["amount"]
        date = order["date"]

        if cid not in features:
            features[cid] = {
                "total_orders": 0,
                "total_amount": 0,
                "avg_amount": 0,
                "last_order_date": date
            }

        features[cid]["total_orders"] += 1
        features[cid]["total_amount"] += amount

        if date > features[cid]["last_order_date"]:
            features[cid]["last_order_date"] = date

    for cid in features:
        f = features[cid]
        f["avg_amount"] = f["total_amount"] / f["total_orders"]

    database["features"] = features

def prediction_service_read(customer_id):
    return database["features"].get(customer_id)

orders_service = []

def order_service_add(order):
    orders_service.append(order)

def order_service_get():
    return orders_service

def feature_service_get_features():
    features = {}
    orders = order_service_get()

    for order in orders:
        cid = order["customer_id"]
        amount = order["amount"]
        date = order["date"]

        if cid not in features:
            features[cid] = {
                "total_orders": 0,
                "total_amount": 0,
                "avg_amount": 0,
                "last_order_date": date
            }

        features[cid]["total_orders"] += 1
        features[cid]["total_amount"] += amount

        if date > features[cid]["last_order_date"]:
            features[cid]["last_order_date"] = date

    for cid in features:
        f = features[cid]
        f["avg_amount"] = f["total_amount"] / f["total_orders"]

    return features

def prediction_service(customer_id):
    features = feature_service_get_features()
    return features.get(customer_id)

class Broker:
    def __init__(self):
        self.topics = {}

    def publish(self, topic, message):
        self.topics.setdefault(topic, []).append(message)

    def get_messages(self, topic):
        return self.topics.get(topic, [])

    def get_latest(self, topic):
        if topic in self.topics and self.topics[topic]:
            return self.topics[topic][-1]
        return None


broker = Broker()

def order_publish(order):
    broker.publish("orders", order)

def feature_compute():
    orders = broker.get_messages("orders")
    features = {}

    for order in orders:
        cid = order["customer_id"]
        amount = order["amount"]
        date = order["date"]

        if cid not in features:
            features[cid] = {
                "total_orders": 0,
                "total_amount": 0,
                "avg_amount": 0,
                "last_order_date": date
            }

        features[cid]["total_orders"] += 1
        features[cid]["total_amount"] += amount

        if date > features[cid]["last_order_date"]:
            features[cid]["last_order_date"] = date

    for cid in features:
        f = features[cid]
        f["avg_amount"] = f["total_amount"] / f["total_orders"]

    broker.publish("features", features)

def prediction_read(customer_id):
    latest = broker.get_latest("features")
    if latest:
        return latest.get(customer_id)
    return None

customers = ["C1", "C2", "C3"]

orders_list = []
for i in range(20):
    order = {
        "customer_id": random.choice(customers),
        "amount": random.randint(10, 100),
        "date": i
    }
    orders_list.append(order)

for o in orders_list:
    order_service_write(o)

feature_service_compute()
db_result = prediction_service_read("C1")

for o in orders_list:
    order_service_add(o)

service_result = prediction_service("C1")

for o in orders_list:
    order_publish(o)

feature_compute()
broker_result = prediction_read("C1")


print("Database:", db_result)
print("Service:", service_result)
print("Broker:", broker_result)

assert db_result == service_result == broker_result

In [ ]:
"""
Task 4: Batch Processing vs Stream Processing
Step 1 — Batch processing: Using the cleaned data from Task 1, write a batch processing function that computes daily aggregate features:

Total orders per day
Total revenue per day
Average order size per day
Number of unique customers per day
Top product per day (by revenue)
Execute it and display the results.

Step 2 — Stream processing: Implement a StreamProcessor class that processes orders one at a time and maintains running statistics:

Running total of orders and revenue
Windowed average (last 50 orders) of order size
Running count of unique customers
Current most popular product (by count in the last 50 orders)
Process the same data from Task 1 through the stream processor, one record at a time. After every 50 records, print the current streaming statistics.

Step 3 — Compare final results: After processing all records through both approaches, compare the final totals
They should match for cumulative statistics (total orders, total revenue)
Write a markdown cell explaining: Why might windowed streaming statistics differ from daily batch statistics even on the same data?
What does each approach tell you that the other does not?
"""

In [ ]:
def daily_agg_feat_safe(orders):
    daily = {}

    for order in orders:
        product = order.get("product_name") or "Unknown"

        try:
            quantity = float(order["quantity"])
        except:
            quantity = 1
        try:
            price = float(order["unit_price"])
        except:
            price = 10

        if quantity <= 0 or price <= 0:
            continue

        try:
            date = datetime.strptime(order["order_date"], "%Y-%m-%d").date()
        except:
            continue

        if date not in daily:
            daily[date] = {
                "total_orders": 0,
                "total_revenue": 0.0,
                "order_sizes": [],
                "customers": set(),
                "product_revenue": {}
            }

        stats = daily[date]
        stats["total_orders"] += 1
        revenue = quantity * price
        stats["total_revenue"] += revenue
        stats["order_sizes"].append(quantity)
        stats["customers"].add(order.get("customer_id", "Unknown"))

        if product not in stats["product_revenue"]:
            stats["product_revenue"][product] = 0.0
        stats["product_revenue"][product] += revenue

    summary = {}
    for date, stats in daily.items():
        avg_order_size = sum(stats["order_sizes"]) / len(stats["order_sizes"])
        top_product = max(stats["product_revenue"], key=stats["product_revenue"].get)
        summary[date] = {
            "total_orders": stats["total_orders"],
            "total_revenue": round(stats["total_revenue"], 2),
            "avg_order_size": round(avg_order_size, 2),
            "unique_customers": len(stats["customers"]),
            "top_product": top_product
        }

    return summary

daily_features = daily_agg_feat_safe(raw_orders)

for i, (date, stats) in enumerate(sorted(daily_features.items())):
    print(date, stats)
    if i >= 4:
        break

print("")

class StreamProcessor:
    def __init__(self, window_size=50):
        self.total_orders = 0
        self.total_revenue = 0.0
        self.unique_customers = set()

        self.window_size = window_size
        self.last_orders = deque(maxlen=window_size)
        self.product_counts = Counter()

    def process_order(self, order):
        product = order.get("product_name") or "Unknown"
        try:
            quantity = float(order["quantity"])
            price = float(order["unit_price"])
        except:
            quantity, price = 1, 10

        if quantity <= 0 or price <= 0:
            return

        try:
            datetime.strptime(order["order_date"], "%Y-%m-%d")
        except:
            return

        self.total_orders += 1
        self.total_revenue += quantity * price
        self.unique_customers.add(order.get("customer_id", "Unknown"))

        if len(self.last_orders) == self.window_size:
            old_order = self.last_orders.popleft()
            self.product_counts[old_order["product_name"]] -= 1
            if self.product_counts[old_order["product_name"]] == 0:
                del self.product_counts[old_order["product_name"]]

        self.last_orders.append({"product_name": product, "quantity": quantity})
        self.product_counts[product] += 1

    def get_stats(self):
        if self.last_orders:
            avg_order_size = sum(o["quantity"] for o in self.last_orders) / len(self.last_orders)
        else:
            avg_order_size = 0

        if self.product_counts:
            top_product = self.product_counts.most_common(1)[0][0]
        else:
            top_product = None

        return {
            "running_total_orders": self.total_orders,
            "running_total_revenue": round(self.total_revenue, 2),
            "running_unique_customers": len(self.unique_customers),
            "windowed_avg_order_size": round(avg_order_size, 2),
            "current_top_product": top_product
        }

processor = StreamProcessor(window_size=50)

for i, order in enumerate(raw_orders, 1):
    processor.process_order(order)
    if i % 50 == 0:
        stats = processor.get_stats()
        print(f"After {i} orders:", stats)

print("")

daily_features = daily_agg_feat_safe(raw_orders)

batch_total_orders = sum(day["total_orders"] for day in daily_features.values())
batch_total_revenue = sum(day["total_revenue"] for day in daily_features.values())

stream_stats = processor.get_stats()
stream_total_orders = stream_stats["running_total_orders"]
stream_total_revenue = stream_stats["running_total_revenue"]

print("Batch totals:")
print("Total orders:", batch_total_orders)
print("Total revenue:", batch_total_revenue)

print("")

print("Stream totals:")
print("Total orders:", stream_total_orders)
print("Total revenue:", stream_total_revenue)

print("")

print("Do totals match?")
print("Orders match:", batch_total_orders == stream_total_orders)
print("Revenue match:", abs(batch_total_revenue - stream_total_revenue) < 0.01)

In [ ]:
def compute_batch_features(orders, customer_id):
    cust_orders = [o for o in orders if o.get("customer_id") == customer_id]

    if not cust_orders:
        return {
            "total_lifetime_orders": 0,
            "avg_order_amount": 0,
            "days_since_first_order": None,
            "most_purchased_category": None
        }

    total_orders = len(cust_orders)
    total_amount = sum(float(o["quantity"]) * float(o["unit_price"]) for o in cust_orders if float(o["quantity"]) > 0 and float(o["unit_price"]) > 0)
    avg_amount = round(total_amount / total_orders, 2)

    dates = []
    for o in cust_orders:
        try:
            d = datetime.strptime(o["order_date"], "%Y-%m-%d")
            dates.append(d)
        except:
            continue

    days_since_first = (datetime(2026,3,5) - min(dates)).days if dates else None

    product_counts = Counter()
    for o in cust_orders:
        prod = o.get("product_name") or "Unknown"
        product_counts[prod] += 1
    most_purchased = product_counts.most_common(1)[0][0] if product_counts else None

    return {
        "total_lifetime_orders": total_orders,
        "avg_order_amount": avg_amount,
        "days_since_first_order": days_since_first,
        "most_purchased_category": most_purchased
    }

def compute_stream_features(orders, customer_id, window=10):
    cust_orders = [o for o in orders if o.get("customer_id") == customer_id]

    if not cust_orders:
        return {
            "recent_order_count": 0,
            "recent_avg_amount": 0,
            "seconds_since_last_order": None,
            "recent_top_category": None
        }

    last_orders = cust_orders[-window:]
    recent_order_count = len(last_orders)

    total_amount = sum(float(o["quantity"]) * float(o["unit_price"]) for o in last_orders if float(o["quantity"]) > 0 and float(o["unit_price"]) > 0)
    recent_avg = round(total_amount / recent_order_count, 2)

    last_date = None
    for o in reversed(last_orders):
        try:
            last_date = datetime.strptime(o["order_date"], "%Y-%m-%d")
            break
        except:
            continue

    seconds_since_last = (datetime(2026,3,5) - last_date).total_seconds() if last_date else None

    product_counts = Counter()
    for o in last_orders:
        prod = o.get("product_name") or "Unknown"
        product_counts[prod] += 1
    recent_top = product_counts.most_common(1)[0][0] if product_counts else None

    return {
        "recent_order_count": recent_order_count,
        "recent_avg_amount": recent_avg,
        "seconds_since_last_order": seconds_since_last,
        "recent_top_category": recent_top
    }

sample_customers = random.sample([o["customer_id"] for o in raw_orders], 5)
combined_features = {}

for cid in sample_customers:
    batch_feats = compute_batch_features(raw_orders, cid)
    stream_feats = compute_stream_features(raw_orders, cid, window=10)
    combined = batch_feats.copy()
    combined.update(stream_feats)
    combined_features[cid] = combined

for cid, feats in combined_features.items():
    print(f"Customer {cid}:")
    pprint(feats)
    print("")

In [ ]:
"""
A ML model can use both batch and stream features to learn about a customer’s past and recent behavior
Batch features show what the customer usually does, while stream features show what they are doing now
For example, batch features alone cant predict a sudden new purchase, and stream features alone cant calculate the customer’s total lifetime value.
"""